# Analyse Quantitative BRVM — Screener Stratégie Mixte

Ce notebook évalue les actions de la **Bourse Régionale des Valeurs Mobilières (BRVM)**  
selon une stratégie **Mixte / Équilibrée** :

| Axe | Indicateur | Poids |
|---|---|---|
| Rentabilité | CAGR | 30 % |
| Dividendes | Dividend Yield | 25 % |
| Risque maîtrisé | Faible Volatilité | 20 % |
| Performance/Risque | Ratio de Sharpe | 15 % |
| Liquidité | Volume moyen 30j | 10 % |

> **Données** : Le système charge automatiquement les fichiers CSV dans `data/raw/`.  
> En leur absence, des données simulées (Geometric Brownian Motion) sont utilisées.  
> Pour télécharger des données réelles : `from src.data.brvm_scraper import scrape_all_tickers`

In [ ]:
import sys
import os

# S'assurer que la racine du projet est dans le PYTHONPATH
PROJECT_ROOT = os.path.abspath('..')
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from src.data.brvm_loader import get_historical_data, get_all_tickers, get_ticker_info
from src.strategies.scoring import evaluate_stock, score_and_rank_stocks

# Style graphique
sns.set_theme(style='darkgrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

print('Imports OK — environnement pret.')

## 1. Chargement et Évaluation de toutes les actions BRVM

In [ ]:
tickers = get_all_tickers()
metrics_list = []

for ticker in tickers:
    df = get_historical_data(ticker)
    m  = evaluate_stock(ticker, df)
    if m:
        metrics_list.append(m)

print(f"Analyse effectuée sur {len(metrics_list)} valeurs BRVM.")

## 2. Classement — Screener Mixte

In [ ]:
ranked_df = score_and_rank_stocks(metrics_list, strategy='balanced')

display_cols = ['Ticker', 'Nom', 'Secteur', 'Score_Global', 'CAGR',
                'Dividend_Yield', 'Volatility', 'Max_Drawdown', 'Sharpe_Ratio',
                'Score_Liquidite']

ranked_df[display_cols].style \
    .background_gradient(subset=['Score_Global'], cmap='RdYlGn') \
    .background_gradient(subset=['Score_Liquidite'], cmap='Blues') \
    .set_caption('Classement BRVM — Stratégie Mixte Équilibrée') \
    .set_table_styles([{'selector': 'caption',
                        'props': [('font-size', '14px'), ('font-weight', 'bold')]}])

## 3. Graphique : Risque vs Rendement Total

In [ ]:
import pandas as pd

plot_df = pd.DataFrame(metrics_list).copy()
plot_df['Total_Return'] = plot_df['CAGR'] + plot_df['Dividend_Yield']

fig, ax = plt.subplots(figsize=(11, 7))

scatter = ax.scatter(
    plot_df['Volatility'],
    plot_df['Total_Return'],
    s=plot_df['Volume_30d'] / plot_df['Volume_30d'].max() * 600 + 80,
    c=plot_df['Dividend_Yield'],
    cmap='YlOrRd',
    alpha=0.85,
    edgecolors='white',
    linewidths=0.8
)

# Labels des tickers
for _, row in plot_df.iterrows():
    ax.annotate(
        row['Ticker'],
        (row['Volatility'], row['Total_Return']),
        xytext=(7, 5), textcoords='offset points',
        fontsize=9, fontweight='bold', color='white'
    )

# Ligne Sharpe = 1 (rendement = risque + taux sans risque)
x_range = [plot_df['Volatility'].min() * 0.8, plot_df['Volatility'].max() * 1.1]
ax.plot(x_range, [v + 0.055 for v in x_range],
        linestyle='--', color='cyan', linewidth=1, alpha=0.6, label='Sharpe = 1')

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label('Dividend Yield', color='white')
cbar.ax.yaxis.set_tick_params(color='white')
plt.setp(cbar.ax.yaxis.get_ticklabels(), color='white')

ax.set_xlabel('Risque — Volatilité Annualisée', labelpad=10)
ax.set_ylabel('Rendement Total Estimé (CAGR + Dividende)', labelpad=10)
ax.set_title('BRVM : Rendement vs Risque par Action\n(Taille = Liquidité | Couleur = Dividende)',
             pad=14, fontsize=13)
ax.xaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.yaxis.set_major_formatter(mticker.PercentFormatter(1.0))
ax.legend(loc='lower right')

plt.tight_layout()
plt.savefig('../data/processed/brvm_risk_return.png', dpi=150, bbox_inches='tight')
plt.show()
print('Graphique sauvegardé dans data/processed/brvm_risk_return.png')

## 4. Sélection des meilleures opportunités

In [ ]:
top3 = ranked_df.head(3)
print('=== TOP 3 BRVM — Stratégie Mixte ===')
for i, (_, row) in enumerate(top3.iterrows(), start=1):
    print(f"\n{'='*45}")
    print(f"  #{i} — {row['Nom']} ({row['Ticker']})")
    print(f"  Secteur      : {row['Secteur']}")
    print(f"  Score Global : {row['Score_Global']}/100")
    print(f"  CAGR         : {row['CAGR']}")
    print(f"  Dividende    : {row['Dividend_Yield']}")
    print(f"  Volatilité   : {row['Volatility']}")
    print(f"  Sharpe       : {row['Sharpe_Ratio']}")

## 5. Backtesting — Portefeuille Top-3 Equal-Weight

Simulation d'un portefeuille **equal-weight** sur les 3 meilleures actions du screener,  
avec rebalancement mensuel. Capital initial : **1 000 000 FCFA**.

In [ ]:
from src.backtest.engine import run_backtest
from src.strategies.scoring import _normalize, BALANCED_WEIGHTS

# Charger les prix pour tous les tickers
all_prices = {t: get_historical_data(t) for t in tickers}

# Reconstruire les scores numériques (valeurs non formatées)
raw_df = pd.DataFrame([m for m in metrics_list if m])
raw_df['Score_Croissance'] = _normalize(raw_df['CAGR'])
raw_df['Score_Dividende']  = _normalize(raw_df['Dividend_Yield'])
raw_df['Score_Risque']     = 1 - _normalize(raw_df['Volatility'])
raw_df['Score_Sharpe']     = _normalize(raw_df['Sharpe_Ratio'])
raw_df['Score_Liquidite']  = _normalize(raw_df['Volume_30d'])
raw_df['Score_Global_num'] = sum(w * raw_df[col] for col, w in BALANCED_WEIGHTS.items()) * 100
scores_dict = dict(zip(raw_df['Ticker'], raw_df['Score_Global_num']))

# Lancer le backtest
result = run_backtest(all_prices, scores_dict, top_n=3, initial_capital=1_000_000)

print('=== RÉSULTATS DU BACKTEST ===')
for k, v in result.summary().items():
    print(f"  {k:<22} : {v}")

# Courbe de valeur du portefeuille
fig, ax = plt.subplots(figsize=(12, 5))
result.equity_curve.plot(ax=ax, color='#00d4aa', linewidth=2)
ax.axhline(1_000_000, color='gray', linestyle='--', linewidth=1, label='Capital initial')
ax.set_title('Évolution du Portefeuille BRVM Top-3 (Rebalancement Mensuel)', fontsize=13)
ax.set_xlabel('Date')
ax.set_ylabel('Valeur du Portefeuille (FCFA)')
ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:,.0f}'))
ax.legend()
plt.tight_layout()
plt.savefig('../data/processed/brvm_equity_curve.png', dpi=150, bbox_inches='tight')
plt.show()